# Train a slot-filling LoRA on Colab (any base, any fold, resumable)

Same recipe as the shipped `slots-lora-3` adapter, with the base model and the held-out shop type as settings in cell 0.

| Run | `BASE` | `LEAVE_OUT` | What it answers |
|---|---|---|---|
| Final Qwen adapter (done: `slots-lora-3.gguf`) | `Qwen/Qwen3-14B` | `''` | – |
| Gemma 4 adapter | `google/gemma-4-12B-it` | `''` | Is Gemma better at the form than Qwen, after the same training? |
| Held-out fold | either | e.g. `'nha_thuoc'` | How the adapter does on a shop type it never saw (proxy for the private test's new types) |
| Held-out shops | either, `LABELS='dev'` | `''` | Odd train shops only, so the fair sets (even shops) are unseen: the clean Gemma-vs-Qwen comparison |

- **GPU:** G4 (RTX PRO 6000, 96 GB), A100 or H100 (≥ 40 GB). Checkpoints go to `MyDrive/zalo-dm-runs/ckpt/<TAG>` every 10 steps; after a disconnect, **Run all** again.
- **Output:** `MyDrive/zalo-dm-runs/adapters/<TAG>.gguf` (+ `.sha256`, PEFT folder).
- Before the Gemma run, compare the *base* models first (no training); train only if Gemma is not worse.
- **Cells 7–9 (optional, `EVAL=True`):** build llama.cpp, then run the adapter on the fair sets (rules off) and save the scores to `MyDrive/zalo-dm-runs/results/fair/`.


In [ ]:
# 0. Settings
BASE = 'google/gemma-4-12B-it'   # or 'Qwen/Qwen3-14B'
LEAVE_OUT = ''                   # '' = all types; or one of: tap_hoa ca_phe_tra_sua vat_lieu_xay_dung nha_thuoc nha_hang_quan_nhau
LABELS = 'all'                   # 'all' = all 30 train shops (the shipped adapter); 'dev' = odd shops only (fair sets stay unseen)
EVAL = True                      # cells 7-9: run the adapter on the fair sets after training
EVAL_ONLY = False                # True: skip cells 4-6 and evaluate an adapter already on Drive (adapters/<ADAPTER>.gguf)
SETS = 'clean noisy para'        # fair sets to run in cell 9
RUN = ''                         # suffix for the output names, e.g. '-r2' to rerun after a code change
FAMILY = 'gemma4' if 'gemma' in BASE.lower() else 'qwen3'
TAG = FAMILY + ('-loto-' + LEAVE_OUT if LEAVE_OUT else '-' + LABELS)
# Gemma 4's checkpoint (gemma4_unified) needs transformers 5.10 or later; Qwen3 keeps the versions slots-lora-3 was trained with.
PIP = 'transformers>=5.10 peft accelerate' if FAMILY == 'gemma4' else 'transformers==4.56.2 peft==0.17.1 accelerate==1.10.1'
import os
os.environ.update(BASE=BASE, LEAVE_OUT=LEAVE_OUT, TAG=TAG, PIP=PIP, HF_DIR='/content/base-hf',
                  LABEL_FILE=f'results/training/slots-v3-{LABELS}.jsonl', FAMILY=FAMILY)
ADAPTER = TAG                    # e.g. 'slots-lora-3' with EVAL_ONLY and BASE='Qwen/Qwen3-14B' for the shipped Qwen adapter
os.environ.update(EVAL_ONLY=str(EVAL_ONLY), ADAPTER=ADAPTER, SETS=SETS, RUN=RUN)
print(TAG, ADAPTER)


In [ ]:
# 1. GPU check
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{torch.cuda.get_device_name(0)}: {vram:.0f} GB')
assert vram >= 39, 'Need an A100/H100 (>= 40 GB). Change the runtime type.'

In [ ]:
# 2. Drive: code, labels and checkpoints live in MyDrive/zalo-dm-runs
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/zalo-dm-runs'
for p in ['source/zalo-dm/scripts/train_slots.py', 'source/zalo-dm/splits/shop_types.json', 'source/zalo-dm/results/training/slots-v3-all.jsonl', 'source/zalo-dm/results/training/slots-v3-dev.jsonl', 'source/train/questions.jsonl']:
    assert os.path.exists(f'{D}/{p}'), f'missing {D}/{p} (upload the current zalo-dm source first)'
print('checkpoint on Drive:', os.listdir(f'{D}/ckpt/{TAG}') if os.path.isdir(f'{D}/ckpt/{TAG}') else 'none (fresh start)')
print('finished adapter on Drive:', os.path.exists(f'{D}/adapters/{TAG}.gguf'))

In [ ]:
%%bash
# 3. Local copies (Drive FUSE is slow for training I/O) and packages
set -e
D=/content/drive/MyDrive/zalo-dm-runs
rsync -a --delete --exclude '.cache' --exclude 'results/cache' --exclude 'results/fair' --exclude 'results/public' $D/source/zalo-dm/ /content/zalo-dm/
rsync -a $D/source/train/ /content/train/
mkdir -p /content/ckpt
[ -d $D/ckpt/$TAG ] && rsync -a --delete $D/ckpt/$TAG/ /content/ckpt/$TAG/ || true
# G4 (RTX PRO 6000 Blackwell, sm_120) needs a CUDA 12.8 build of torch; older wheels cannot run on it.
python -c "import torch,sys; sys.exit(0 if torch.cuda.get_device_capability()[0] < 12 or tuple(map(int, (torch.version.cuda or '0.0').split('.')[:2])) >= (12, 8) else 1)" \
  || pip install -q torch --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -1
pip install -q $PIP safetensors sentencepiece "huggingface_hub[cli]" 2>&1 | tail -2
# Colab ships torchao 0.10; recent peft refuses any torchao below 0.16 even when it is unused, so remove it.
pip uninstall -y -q torchao 2>&1 | tail -1 || true
python -c "import torch, transformers, peft; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0)); print('transformers', transformers.__version__, 'peft', peft.__version__)"

In [ ]:
%%bash
# 4. Base model weights (~24-30 GB)
[ "$EVAL_ONLY" = True ] && { echo "skipped (EVAL_ONLY)"; exit 0; }
hf download $BASE --local-dir $HF_DIR 2>&1 | tail -1
du -sh $HF_DIR

In [ ]:
%%bash
# 5. Train (resumes automatically). Re-run after a disconnect (after cells 0-4).
[ "$EVAL_ONLY" = True ] && { echo "skipped (EVAL_ONLY)"; exit 0; }
cd /content/zalo-dm
D=/content/drive/MyDrive/zalo-dm-runs
HOOK="rm -rf $D/ckpt/$TAG.tmp && mkdir -p $D/ckpt && cp -r /content/ckpt/$TAG $D/ckpt/$TAG.tmp && rm -rf $D/ckpt/$TAG && mv $D/ckpt/$TAG.tmp $D/ckpt/$TAG"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
python -m scripts.train_slots --data $LABEL_FILE --model $HF_DIR --leave-out "$LEAVE_OUT" \
  --output /content/$TAG-hf --rank 16 --epochs 2 --lr 1e-4 --batch-size 0 --grad-checkpoint \
  --checkpoint /content/ckpt/$TAG --save-every 10 --on-save "$HOOK" --max-hours 100 2>&1 | grep --line-buffered -vE 'Loading checkpoint shards|use_cache'

In [ ]:
%%bash
# 6. Convert to GGUF (llama.cpp's converter, CPU only) and save to Drive
[ "$EVAL_ONLY" = True ] && { echo "skipped (EVAL_ONLY)"; exit 0; }
set -e
D=/content/drive/MyDrive/zalo-dm-runs
[ -d /content/llama.cpp ] || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
pip install -q /content/llama.cpp/gguf-py mistral-common 2>&1 | tail -1
cd /content
PYTHONPATH=/content/llama.cpp python /content/llama.cpp/convert_lora_to_gguf.py --outtype f16 --base $HF_DIR \
  --outfile /content/$TAG.gguf /content/$TAG-hf 2>&1 | tail -2
sha256sum $TAG.gguf | tee $TAG.gguf.sha256
mkdir -p $D/adapters
cp $TAG.gguf $TAG.gguf.sha256 $D/adapters/
cp -r /content/$TAG-hf $D/adapters/
ls -la $D/adapters

Done: `MyDrive/zalo-dm-runs/adapters/<TAG>.gguf` (+ `.sha256`). Cells 7–9 evaluate it on the fair sets.

## Evaluation on the fair sets (optional)

Rules off, so every question goes to the model: the same runs as `results/fair/v3final-model-*` (Qwen + slots-lora-3). The three sets run in parallel (~1.5 h on an A100). A finished set already on Drive is skipped.


In [ ]:
%%bash -s "$EVAL"
# 7. llama.cpp server with CUDA (~10 min)
[ "$1" = True ] || { echo skipped; exit 0; }
set -e
[ -d /content/llama.cpp ] || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
cd /content/llama.cpp
[ -x build/bin/llama-server ] || { cmake -S . -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build --target llama-server -j 12 2>&1 | grep -E 'error|FAILED' || true; }
mkdir -p /content/zalo-dm/.runtime-cuda
cp -L build/bin/llama-server /content/zalo-dm/.runtime-cuda/ && find build -name 'lib*.so*' -exec cp -P {} /content/zalo-dm/.runtime-cuda/ \;
/content/zalo-dm/.runtime-cuda/llama-server --version 2>&1 | tail -1


In [ ]:
# 8. Model files and the runner's one dependency
import subprocess
if EVAL:
    GGUF = {'gemma4': ('ggml-org/gemma-4-12B-it-GGUF', 'gemma-4-12B-it-Q8_0.gguf'), 'qwen3': ('Qwen/Qwen3-14B-GGUF', 'Qwen3-14B-Q5_K_M.gguf')}[FAMILY]
    print(subprocess.run(f'hf download {GGUF[0]} {GGUF[1]} --local-dir /content/gguf 2>&1 | tail -1; pip install -q -e /content/zalo-dm 2>&1 | tail -1', shell=True, capture_output=True, text=True).stdout)
    MODEL_GGUF = f'/content/gguf/{GGUF[1]}'
    os.environ['MODEL_GGUF'] = MODEL_GGUF  # cell 9 reads it: /content/gguf may also hold the other family's file
    ADAPTER_FILE = f'/content/{ADAPTER}.gguf'
    if not os.path.exists(ADAPTER_FILE):
        subprocess.run(['cp', f'{D}/adapters/{ADAPTER}.gguf', ADAPTER_FILE], check=True)
    print(subprocess.run(['sha256sum', ADAPTER_FILE], capture_output=True, text=True).stdout)


In [ ]:
%%bash -s "$EVAL" "$ADAPTER"
# 9. The adapter on the fair sets in SETS, rules off, in parallel; results and scores go to Drive
[ "$1" = True ] || { echo skipped; exit 0; }
cd /content/zalo-dm
D=/content/drive/MyDrive/zalo-dm-runs/results/fair
G=$MODEL_GGUF
[ -f "$G" ] || { echo "no model file $G: run cell 8"; exit 1; }
python -m scripts.fair_test prepare > /dev/null
for s in $SETS; do
  N=$2-model-$s$RUN
  if [ -f $D/$N/submission.jsonl ]; then echo "$N already on Drive"; continue; fi
  ( python -m scripts.run_metric_layer --data ../train --questions results/fair/$s.jsonl --rules off --rewrite on --sql-fallback on \
      --model $G --lora /content/$2.gguf --runtime .runtime-cuda --databases .cache/metrics-$s --output results/fair/$N > /content/$N.log 2>&1 \
    && python -m scripts.fair_test score results/fair/$N > results/fair/$N/score.txt 2>&1 \
    && mkdir -p $D && cp -r results/fair/$N $D/ && cat results/fair/$N/score.txt \
    || { echo "$N FAILED, last log lines:"; tail -20 /content/$N.log; } ) &
  sleep 60
done
wait
echo ALL_DONE
